In [1]:
from chemical_engine import ChemicalSpecies, Reaction, ChemicalSystem
from visualisation import create_interface
import numpy as np

R_J_MOL_K = 8.314
water = ChemicalSpecies("H2O", species_type='pool', phase='solvent', density=1000.0, molar_mass=18.015)

# ions
proton = ChemicalSpecies("H3O+", phase='aqueous', charge=1)
hydroxide = ChemicalSpecies("OH-", phase='aqueous', charge=-1)
sodium = ChemicalSpecies("Na+", phase='aqueous', charge=1) # spectator ion

weak_acid = ChemicalSpecies("HA", phase='aqueous')
conj_base = ChemicalSpecies("A-", phase='aqueous', charge=-1)

weak_base = ChemicalSpecies("NH3", phase='aqueous')
conj_acid = ChemicalSpecies("NH4+", phase='aqueous', charge=+1)

species_list = [water, proton, hydroxide, sodium, weak_acid, conj_base, weak_base, conj_acid]

T_ref = 298.0
RT = R_J_MOL_K * T_ref

# water ionisation
# 2 H2O <-> H3O+ + OH-
Ea_r_w = 12000.0
k_r_w = 1.3e11
A_r_w = k_r_w / np.exp(-Ea_r_w / RT)

Ea_f_w = 68000.0
k_f_w = 4.21e-7
A_f_w = k_f_w / np.exp(-Ea_f_w / RT)

# weak acid dissociation
Ea_r_a = 10000.0
k_r_a = 5.0e10
A_r_a = k_r_a / np.exp(-Ea_r_a / RT)

Ea_f_a = 10000.0 
k_f_a = 1.56e4
A_f_a = k_f_a / np.exp(-Ea_f_a / RT)

# weak base dissociation
Ea_r_b = 10000.0
k_r_b = 3.0e10
A_r_b = k_r_b / np.exp(-Ea_r_b / RT)

Ea_f_b = 10000.0
k_f_b = 9.6e3
A_f_b = k_f_b / np.exp(-Ea_f_b / RT)

r1_fwd = Reaction({'H2O': 2}, {'H3O+': 1, 'OH-': 1}, A=A_f_w, Ea=Ea_f_w)
r1_rev = Reaction({'H3O+': 1, 'OH-': 1}, {'H2O': 2}, A=A_r_w, Ea=Ea_r_w)

r2_fwd = Reaction({'HA': 1, 'H2O': 1}, {'A-': 1, 'H3O+': 1}, A=A_f_a, Ea=Ea_f_a)
r2_rev = Reaction({'A-': 1, 'H3O+': 1}, {'HA': 1, 'H2O': 1}, A=A_r_a, Ea=Ea_r_a)

r3_fwd = Reaction({'NH3': 1, 'H2O': 1}, {'NH4+': 1, 'OH-': 1}, A=A_f_b, Ea=Ea_f_b)
r3_rev = Reaction({'NH4+': 1, 'OH-': 1}, {'NH3': 1, 'H2O': 1}, A=A_r_b, Ea=Ea_r_b)

reactions = [r1_fwd, r1_rev, r2_fwd, r2_rev, r3_fwd, r3_rev]

initial_moles = {
    'H2O': 0.0, # will be auto-calculated
    'HA': 0.1,  # 0.1 moles in 1L = 0.1M
    'A-': 0.0,
    'H3O+': 0.0,
    'OH-': 0.0,
    'Na+': 0.0
}

V_initial = 1.0 # L
T_initial = 298.0 

buffer_system = ChemicalSystem(
    species_list, reactions, initial_moles, V_initial, T_initial,
    method='BDF', rtol=1e-8, atol=1e-12, system_type='acid_base'
)

print(f"pKa {4.76}")
print(f"pH expected: { -np.log10(np.sqrt(1.74e-5 * 0.1)) :.2f}") 

create_interface(buffer_system)

pKa 4.76
pH expected: 2.88


# Polyprotic Titrations
* $2H_2O(l)\rightleftharpoons H_3O^+(aq)+OH^-(aq)$

A diprotic acid ($H_2A$) possesses two ionisable protons. It does not lose them simultaneously. It undergoes stepwise dissociation, governed by two distinct equilibrium constants.
* $H_2A(aq)+H_2O(l)\rightleftharpoons HA^-(aq)+H_3O^+(aq)$ ($pK_{a1}=3.0$)
    * This is the stronger proton and, initially, this equilibrium dominates. The species $H_2A$ acts as the acid.
* $HA^-(aq)+H_2O(l)\rightleftharpoons A^{2-}(aq)+H_3O^+(aq)$ ($pK_{a2}=7.0$)
    * This is the weaker proton. Here, the intermediate ion $HA^-$ acts as a weak acid.
    * It is much harder to remove a proton from a negatively charged ion ($HA^-$) than a neutral molecules ($H_2A$), so $pk_{a2}>pk_{a1}$. 

### Stiffness
This system is stiff as it involves three different timescales:
* Water autoionisation.
* Deprotonation of $H_2A$.
* Titration injection (100 s).

## Observations
When titrating $H_2A$ with a strong base ($OH^-$), there are two distinct phases corresponding to the two equilibria.
* **First Buffer Region ($pH\approx pK_{a1}$)**
    * $[H_2A]\approx[HA^-]$
    * The system buffers at around pH 3.0.
* **First Equivalence Point**
    * All $H_2A$ has been converted to $HA^-$. 
    * Sharp rise in pH.
    * Here, $HA^-$ is amphoteric and the pH at this point is $\frac{pK_{a1}+pK_{a2}}{2}\approx5.0$.
* **Second Buffer Region ($pH\approx pK_{a2}$)**
    * $[HA^-]\approx[A^{2-}]$
    * The system buffers again, now around pH 7.0. 
* **Second Equivalence Point**
    * All $HA^-$ has been converted to $A^{2-}$.
    * A second sharp rise in pH.

In [2]:
water = ChemicalSpecies("H2O", species_type='pool', phase='solvent', density=1000.0, molar_mass=18.015)

# ions
proton = ChemicalSpecies("H3O+", phase='aqueous', charge=1)
hydroxide = ChemicalSpecies("OH-", phase='aqueous', charge=-1)
sodium = ChemicalSpecies("Na+", phase='aqueous', charge=1) # spectator

# diprotic Weak Acid System (H2A -> HA- -> A2-)
acid_full = ChemicalSpecies("H2A", phase='aqueous', charge=0) 
acid_inter = ChemicalSpecies("HA-", phase='aqueous', charge=-1)
acid_final = ChemicalSpecies("A2-", phase='aqueous', charge=-2)

species_list = [water, proton, hydroxide, sodium, acid_full, acid_inter, acid_final]

T_ref = 298.0
RT = R_J_MOL_K * T_ref
H2O_conc = 55.51 # Molar concentration of water

# water autoionisation
Ea_r_w = 12000.0
k_r_w = 1.3e11 
A_r_w = k_r_w / np.exp(-Ea_r_w / RT)

Ea_f_w = 68000.0
k_f_w = 4.21e-7 # derived from Kw and [H2O]
A_f_w = k_f_w / np.exp(-Ea_f_w / RT)


pKa1 = 3.0
Ka1 = 10**(-pKa1)

k_r_1 = 5.0e10 
Ea_r_1 = 10000.0
A_r_1 = k_r_1 / np.exp(-Ea_r_1 / RT)

# forward
Kc_1 = Ka1 / H2O_conc
k_f_1 = Kc_1 * k_r_1
Ea_f_1 = 10000.0 
A_f_1 = k_f_1 / np.exp(-Ea_f_1 / RT)


pKa2 = 7.0
Ka2 = 10**(-pKa2)

# reverse (A2- + H3O+ -> HA-)
k_r_2 = 5.0e10
Ea_r_2 = 10000.0
A_r_2 = k_r_2 / np.exp(-Ea_r_2 / RT)

# forward (HA- dissociation)
Kc_2 = Ka2 / H2O_conc
k_f_2 = Kc_2 * k_r_2
Ea_f_2 = 10000.0
A_f_2 = k_f_2 / np.exp(-Ea_f_2 / RT)

# water auto-ionisation
r_w_fwd = Reaction({'H2O': 2}, {'H3O+': 1, 'OH-': 1}, A=A_f_w, Ea=Ea_f_w)
r_w_rev = Reaction({'H3O+': 1, 'OH-': 1}, {'H2O': 2}, A=A_r_w, Ea=Ea_r_w)

# acid Step 1
r1_fwd = Reaction({'H2A': 1, 'H2O': 1}, {'HA-': 1, 'H3O+': 1}, A=A_f_1, Ea=Ea_f_1)
r1_rev = Reaction({'HA-': 1, 'H3O+': 1}, {'H2A': 1, 'H2O': 1}, A=A_r_1, Ea=Ea_r_1)

# acid Step 2
r2_fwd = Reaction({'HA-': 1, 'H2O': 1}, {'A2-': 1, 'H3O+': 1}, A=A_f_2, Ea=Ea_f_2)
r2_rev = Reaction({'A2-': 1, 'H3O+': 1}, {'HA-': 1, 'H2O': 1}, A=A_r_2, Ea=Ea_r_2)

reactions = [r_w_fwd, r_w_rev, r1_fwd, r1_rev, r2_fwd, r2_rev]

initial_moles = {
    'H2O': 0.0,
    'H2A': 0.1,  # 0.1 M initial acid
    'HA-': 0.0,
    'A2-': 0.0,
    'H3O+': 0.0,
    'OH-': 0.0,
    'Na+': 0.0
}

V_initial = 1.0 # 1.0 L
T_initial = 298.0 

diprotic_system = ChemicalSystem(
    species_list, reactions, initial_moles, V_initial, T_initial,
    method='BDF', rtol=1e-8, atol=1e-12,
    system_type='acid_base'
)

create_interface(diprotic_system)

## Triprotic Acid
* **Step 1:** $pK_{a1}\approx2.1$
* **Step 2:** $pK_{a2}\approx7.2$
* **Step 3:** $pK_{a3}\approx12.3$

In [3]:
water = ChemicalSpecies("H2O", species_type='pool', phase='solvent', density=1000.0, molar_mass=18.015)

# ions
proton = ChemicalSpecies("H3O+", phase='aqueous', charge=1)
hydroxide = ChemicalSpecies("OH-", phase='aqueous', charge=-1)
sodium = ChemicalSpecies("Na+", phase='aqueous', charge=1)

acid_3 = ChemicalSpecies("H3A", phase='aqueous', charge=0) 
acid_2 = ChemicalSpecies("H2A-", phase='aqueous', charge=-1)
acid_1 = ChemicalSpecies("HA2-", phase='aqueous', charge=-2)
acid_0 = ChemicalSpecies("A3-", phase='aqueous', charge=-3)

species_list = [water, proton, hydroxide, sodium, acid_3, acid_2, acid_1, acid_0]

T_ref = 298.0
RT = R_J_MOL_K * T_ref
H2O_conc = 55.51

# water
Ea_r_w = 12000.0
k_r_w = 1.3e11 
A_r_w = k_r_w / np.exp(-Ea_r_w / RT)

Ea_f_w = 68000.0
k_f_w = 4.21e-7
A_f_w = k_f_w / np.exp(-Ea_f_w / RT)

# step 1 (pKa1 = 2.1)
Ka1 = 10**(-2.1)
Kc_1 = Ka1 / H2O_conc
k_r_1 = 5.0e10 
Ea_r_1 = 10000.0
A_r_1 = k_r_1 / np.exp(-Ea_r_1 / RT)
k_f_1 = Kc_1 * k_r_1
Ea_f_1 = 10000.0
A_f_1 = k_f_1 / np.exp(-Ea_f_1 / RT)

# step 2 (pKa2 = 7.2)
Ka2 = 10**(-7.2)
Kc_2 = Ka2 / H2O_conc
k_r_2 = 5.0e10
Ea_r_2 = 10000.0
A_r_2 = k_r_2 / np.exp(-Ea_r_2 / RT)
k_f_2 = Kc_2 * k_r_2
Ea_f_2 = 10000.0
A_f_2 = k_f_2 / np.exp(-Ea_f_2 / RT)

# step 3 (pKa3 = 12.3)
Ka3 = 10**(-12.3)
Kc_3 = Ka3 / H2O_conc
k_r_3 = 5.0e10
Ea_r_3 = 10000.0
A_r_3 = k_r_3 / np.exp(-Ea_r_3 / RT)
k_f_3 = Kc_3 * k_r_3
Ea_f_3 = 10000.0
A_f_3 = k_f_3 / np.exp(-Ea_f_3 / RT)


# reactions
r_w_fwd = Reaction({'H2O': 2}, {'H3O+': 1, 'OH-': 1}, A=A_f_w, Ea=Ea_f_w)
r_w_rev = Reaction({'H3O+': 1, 'OH-': 1}, {'H2O': 2}, A=A_r_w, Ea=Ea_r_w)

r1_fwd = Reaction({'H3A': 1, 'H2O': 1}, {'H2A-': 1, 'H3O+': 1}, A=A_f_1, Ea=Ea_f_1)
r1_rev = Reaction({'H2A-': 1, 'H3O+': 1}, {'H3A': 1, 'H2O': 1}, A=A_r_1, Ea=Ea_r_1)

r2_fwd = Reaction({'H2A-': 1, 'H2O': 1}, {'HA2-': 1, 'H3O+': 1}, A=A_f_2, Ea=Ea_f_2)
r2_rev = Reaction({'HA2-': 1, 'H3O+': 1}, {'H2A-': 1, 'H2O': 1}, A=A_r_2, Ea=Ea_r_2)

r3_fwd = Reaction({'HA2-': 1, 'H2O': 1}, {'A3-': 1, 'H3O+': 1}, A=A_f_3, Ea=Ea_f_3)
r3_rev = Reaction({'A3-': 1, 'H3O+': 1}, {'HA2-': 1, 'H2O': 1}, A=A_r_3, Ea=Ea_r_3)

reactions = [r_w_fwd, r_w_rev, r1_fwd, r1_rev, r2_fwd, r2_rev, r3_fwd, r3_rev]

initial_moles = {
    'H2O': 0.0,
    'H3A': 0.1, 
    'H2A-': 0.0,
    'HA2-': 0.0,
    'A3-': 0.0,
    'H3O+': 0.0,
    'OH-': 0.0,
    'Na+': 0.0
}

triprotic_system = ChemicalSystem(
    species_list, reactions, initial_moles, 1.0, 298.0,
    method='BDF', rtol=1e-5, atol=1e-7, system_type='acid_base'
)

create_interface(triprotic_system)